# KCV-TTS Finetuning di Kaggle (2x T4, DDP, W&B)
Notebook ini menyiapkan pipeline penuh: venv, install dependency (termasuk `causal-conv1d` dan `mamba-ssm`), clone repo, prepare dataset CSV -> Arrow, lalu finetuning dengan 2 GPU T4 menggunakan `accelerate` + W&B.

## 1) Isi Konfigurasi
Ubah nilai di cell berikut sesuai repo dan path dataset kamu di Kaggle.

In [ ]:
import os
from pathlib import Path

# ===== WAJIB DIISI =====
REPO_URL = "https://github.com/<username>/<repo>.git"
REPO_BRANCH = "main"
WANDB_API_KEY = "PASTE_WANDB_API_KEY_DI_SINI"
WANDB_ENTITY = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "kcvanguard"

# CSV metadata sumber di Kaggle (header: audio_file|text, path audio absolut)
# Contoh: /kaggle/input/your-dataset/metadata.csv
SRC_METADATA_CSV = "/kaggle/input/your-dataset/metadata.csv"

# Nama dataset hasil prep (akan jadi data/{DATASET_NAME}_pinyin)
DATASET_NAME = "datasetku"

# Output training
MODEL_NAME = "F5TTS_MAMBA_kaggle_full"
TARGET_SAMPLE_RATE = 24000

# Training optimization untuk 2x T4
BATCH_SIZE_PER_GPU = 512
MAX_SAMPLES = 2
GRAD_ACCUM = 4
LEARNING_RATE = 1.0e-5
NUM_WARMUP_UPDATES = 200
EPOCHS = 30
NUM_WORKERS = 4

# Path kerja
WORKDIR = Path('/kaggle/working')
VENV_DIR = WORKDIR / '.venv'
REPO_DIR = WORKDIR / 'kcv-tts'

assert WANDB_API_KEY != "PASTE_WANDB_API_KEY_DI_SINI", "Isi WANDB_API_KEY dulu"
assert REPO_URL != "https://github.com/<username>/<repo>.git", "Isi REPO_URL dulu"
assert Path(SRC_METADATA_CSV).exists(), f"metadata tidak ditemukan: {SRC_METADATA_CSV}"

# Export ke environment agar tersedia di semua cell %%bash
os.environ['REPO_URL'] = REPO_URL
os.environ['REPO_BRANCH'] = REPO_BRANCH
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_ENTITY'] = WANDB_ENTITY
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
os.environ['WANDB_MODE'] = 'online'
os.environ['SRC_METADATA_CSV'] = SRC_METADATA_CSV
os.environ['DATASET_NAME'] = DATASET_NAME
os.environ['MODEL_NAME'] = MODEL_NAME
os.environ['TARGET_SAMPLE_RATE'] = str(TARGET_SAMPLE_RATE)
os.environ['BATCH_SIZE_PER_GPU'] = str(BATCH_SIZE_PER_GPU)
os.environ['MAX_SAMPLES'] = str(MAX_SAMPLES)
os.environ['GRAD_ACCUM'] = str(GRAD_ACCUM)
os.environ['LEARNING_RATE'] = str(LEARNING_RATE)
os.environ['NUM_WARMUP_UPDATES'] = str(NUM_WARMUP_UPDATES)
os.environ['EPOCHS'] = str(EPOCHS)
os.environ['NUM_WORKERS'] = str(NUM_WORKERS)

print('Config OK')
print('SRC_METADATA_CSV =', SRC_METADATA_CSV)

In [ ]:
!nvidia-smi

## 2) Build Venv + Clone Repo + Install Dependencies

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working

python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip setuptools wheel

# Clone repo
if [ -d kcv-tts ]; then rm -rf kcv-tts; fi
git clone --depth 1 --branch "${REPO_BRANCH:-main}" "${REPO_URL}" kcv-tts

cd kcv-tts

# Install project package
pip install -e .

# Install Kaggle stack pinned to torch 2.10
pip install -r requirements-kaggle-torch210.txt

python -V
pip list | grep -E "torch|accelerate|wandb|mamba-ssm|causal-conv1d"

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/.venv/bin/activate
python - <<'PY'
import os
import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])
print('W&B login OK')
PY

## 3) Prepare Dataset (CSV -> Arrow)

In [ ]:
import csv
from pathlib import Path

src = Path(SRC_METADATA_CSV)
dst = Path('/kaggle/working/metadata_abs.csv')

with src.open('r', encoding='utf-8-sig', newline='') as fin, dst.open('w', encoding='utf-8', newline='') as fout:
    r = csv.reader(fin, delimiter='|')
    w = csv.writer(fout, delimiter='|', lineterminator='\n')
    header = next(r, None)
    if not header or header[0].strip() != 'audio_file' or header[1].strip() != 'text':
        raise ValueError('Header CSV harus: audio_file|text')
    w.writerow(['audio_file', 'text'])
    n = 0
    for row in r:
        if len(row) < 2:
            continue
        audio = row[0].strip()
        text = row[1].strip()
        if not audio or not text:
            continue
        p = Path(audio).expanduser()
        if not p.is_absolute():
            # Jika relatif, jadikan relatif terhadap folder CSV sumber
            p = (src.parent / p).resolve()
        if p.exists():
            w.writerow([p.as_posix(), text])
            n += 1

print('metadata_abs.csv rows =', n)
print('saved at', dst)

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/.venv/bin/activate
cd /kaggle/working/kcv-tts

python src/f5_tts/train/datasets/prepare_csv_wavs.py /kaggle/working/metadata_abs.csv /kaggle/working/kcv-tts/data/${DATASET_NAME}_pinyin --workers 8
ls -lah /kaggle/working/kcv-tts/data/${DATASET_NAME}_pinyin

## 4) Download Pretrained Checkpoint ke Save Dir (untuk Finetune)

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/.venv/bin/activate
cd /kaggle/working/kcv-tts

CKPT_DIR="ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
mkdir -p "$CKPT_DIR"

python - <<'PY'
from cached_path import cached_path
import shutil
from pathlib import Path
import os

model_name = os.environ.get('MODEL_NAME', 'F5TTS_MAMBA_kaggle_full')
dataset_name = os.environ.get('DATASET_NAME', 'datasetku')
ckpt_dir = Path(f'/kaggle/working/kcv-tts/ckpts/{model_name}_vocos_pinyin_{dataset_name}')
ckpt_dir.mkdir(parents=True, exist_ok=True)

src = Path(str(cached_path('hf://SWivid/F5-TTS/F5TTS_v1_Base/model_1250000.safetensors')))
dst = ckpt_dir / 'pretrained_model_1250000.safetensors'
if not dst.exists():
    shutil.copy2(src, dst)
print('pretrained ckpt ready at', dst)
PY

## 5) Finetune Full di 2x T4 (DDP) + W&B

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/.venv/bin/activate
cd /kaggle/working/kcv-tts

export WANDB_API_KEY="${WANDB_API_KEY}"
export WANDB_ENTITY="${WANDB_ENTITY}"
export WANDB_PROJECT="${WANDB_PROJECT}"
export WANDB_MODE=online

export CUDA_VISIBLE_DEVICES=0,1

# Catatan:
# - ++model.arch.use_mamba=true mengaktifkan selective SSM path.
# - save_dir tetap sama agar auto-resume dari pretrained_/model_last.

accelerate launch --multi_gpu --num_processes 2 --mixed_precision=fp16 \
  src/f5_tts/train/train.py \
  --config-name F5TTS_SANITY_HYBRID_1K.yaml \
  ++datasets.name=${DATASET_NAME} \
  ++datasets.batch_size_per_gpu=${BATCH_SIZE_PER_GPU} \
  ++datasets.batch_size_type=frame \
  ++datasets.max_samples=${MAX_SAMPLES} \
  ++datasets.num_workers=${NUM_WORKERS} \
  ++optim.epochs=${EPOCHS} \
  ++optim.learning_rate=${LEARNING_RATE} \
  ++optim.num_warmup_updates=${NUM_WARMUP_UPDATES} \
  ++optim.grad_accumulation_steps=${GRAD_ACCUM} \
  ++optim.max_grad_norm=1.0 \
  ++optim.bnb_optimizer=True \
  ++ckpts.logger=wandb \
  ++ckpts.wandb_project=${WANDB_PROJECT} \
  ++ckpts.wandb_run_name=${MODEL_NAME}-2xT4 \
  ++ckpts.log_samples=False \
  ++ckpts.save_per_updates=200 \
  ++ckpts.last_per_updates=100 \
  ++ckpts.keep_last_n_checkpoints=3 \
  ++model.name=${MODEL_NAME} \
  ++model.tokenizer=pinyin \
  ++model.mel_spec.target_sample_rate=${TARGET_SAMPLE_RATE} \
  ++model.arch.dim=512 \
  ++model.arch.depth=10 \
  ++model.arch.heads=8 \
  ++model.arch.text_dim=512 \
  ++model.arch.checkpoint_activations=True \
  ++model.arch.use_mamba=true \
  ++model.arch.mamba_layers=[2,3,6,7]

## 6) Lokasi Checkpoint dan Quick Inference (Opsional)

In [ ]:
from pathlib import Path
ckpt_dir = Path(f'/kaggle/working/kcv-tts/ckpts/{MODEL_NAME}_vocos_pinyin_{DATASET_NAME}')
print('Checkpoint dir:', ckpt_dir)
for p in sorted(ckpt_dir.glob('model*.pt'))[-5:]:
    print(p)

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/.venv/bin/activate
cd /kaggle/working/kcv-tts

# Contoh quick inference setelah model_last.pt tersedia.
# Ganti ref_audio/ref_text sesuai data kamu.
MODEL_CFG="$(ls -1t ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}/*/.hydra/config.yaml | head -n 1)"
python src/f5_tts/infer/infer_cli.py \
  -m F5TTS_Small \
  -mc "$MODEL_CFG" \
  -p ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}/model_last.pt \
  -v data/${DATASET_NAME}_pinyin/vocab.txt \
  -r /kaggle/input/your-dataset/example.wav \
  -s "Ini teks referensi." \
  -t "Halo semua kenalin Aku Sedang Makan" \
  -o /kaggle/working \
  -w infer_kaggle.wav \
  --device cuda --nfe_step 24 --cfg_strength 2.0